# Reservoir Audio

Reservoir Audio cuts a sound into chunks, learns the order they come in with a quantum reservoir, and strings them into a new arrangement. Train once, then generate as many takes as you like from the trained model. (The API names it QRC Audio.)

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

ENGINE = "qrc-audio-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "qrc-audio-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
import io
import wave

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display


def samples(wav):
    """A WAV file (path or bytes) as (float samples in [-1, 1], one column per channel; sample rate)."""
    with wave.open(wav if isinstance(wav, str) else io.BytesIO(wav)) as w:
        width, channels, rate = w.getsampwidth(), w.getnchannels(), w.getframerate()
        raw = np.frombuffer(w.readframes(w.getnframes()), np.uint8)
    if width == 3:  # 24-bit: pad each sample to 32 bits
        raw, width = np.pad(raw.reshape(-1, 3), ((0, 0), (1, 0))).reshape(-1), 4
    x = raw.view(f"<i{width}").astype(np.float32) / 2 ** (8 * width - 1)
    return x.reshape(-1, channels), rate


def player(x, rate):
    """A player for float samples (n, or n × channels): a 128 kbps MP3 if lameenc is installed (small, full
    bandwidth), else a WAV at half the rate."""
    x = np.clip(np.asarray(x, np.float32), -1, 1)
    try:
        import lameenc
    except ImportError:
        x, rate = x[::2], rate // 2
        return Audio(x.T if x.ndim == 2 else x, rate=rate, normalize=False)
    enc = lameenc.Encoder()
    enc.set_bit_rate(128)
    enc.set_in_sample_rate(rate)
    enc.set_channels(x.shape[1] if x.ndim == 2 else 1)
    enc.set_quality(2)
    audio = Audio(bytes(enc.encode((x * 32767).astype("<i2").tobytes()) + enc.flush()))
    audio.mimetype = "audio/mpeg"
    return audio


def listen(clips, play=True, stereo=False, height=1.1):
    """Plot and play {label: WAV path or bytes}, one row each. Players are mono unless `stereo`."""
    fig, axes = plt.subplots(len(clips), 1, figsize=(9, height * len(clips)), sharex=True, squeeze=False)
    for ax, (label, wav) in zip(axes[:, 0], clips.items()):
        x, sr = samples(wav)
        t = np.arange(len(x)) / sr
        for c in range(x.shape[1]):
            ax.plot(t, x[:, c], lw=0.4, alpha=0.8)
        ax.set_ylim(-1, 1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
    axes[-1, 0].set_xlabel("seconds")
    plt.show()
    for label, wav in clips.items() if play else ():
        x, sr = samples(wav)
        print(label)
        display(player(x if stereo else x.mean(axis=1), sr))

def output_asset(job, slot):
    """One output file of a finished job, as an Asset you can pass straight into another job's files."""
    out = next(o for o in job.outputs if o.slot == slot)
    return client.assets.get(out.output_asset_id)


BAR = 2.5  # seconds: one bar of loop.wav (96 bpm)
CHORDS = ["Am", "F", "C", "G", "Dm", "F", "E", "Am"]  # the chord of each bar of loop.wav


def progression(remix, source="loop.wav", chunk_seconds=BAR):
    """The chords a remix plays: each chunk matched to the closest bar of the source."""
    x, rate = samples(source)
    y, _ = samples(remix)
    n = int(chunk_seconds * rate)
    bars = [x[i:i + n, 0] for i in range(0, len(x) - n + 1, n)]
    order = [int(np.argmin([np.sum((y[i:i + n, 0] - b) ** 2) for b in bars])) for i in range(0, len(y) - n + 1, n)]
    return " | ".join(CHORDS[k] for k in order)

## Try it

`loop.wav` beside this notebook is 8 bars at 96 bpm built from Moth's QEcho sample pack: a Rhodes, bass and drum groove over the chords Am, F, C, G, Dm, F, E, Am, one chord per bar. Each bar is self-contained, so with `chunk_seconds` set to one bar (2.5 s) every reordering stays in time and in key: the reservoir writes new chord progressions. The line under the players names the chords the remix plays.

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "chunk_seconds": 2.5,
        "length": 8,
        "quality": "fast",
        "variation": 1,
        "loop": True,
        "crossfade": 10,
        "seed": 7,
    },
    files={
        "audio": "loop.wav",
    },
)
listen({"loop": "loop.wav", "remix": job.output_bytes("result")})
print("loop: ", " | ".join(CHORDS))
print("remix:", progression(job.output_bytes("result")))

Keep the trained `model` and the chunk `vocabulary` from that run, and generate another take from them: no training this time.

In [ ]:
model, vocabulary = output_asset(job, "model"), output_asset(job, "vocabulary")

job = client.jobs.run(engine=ENGINE, params={"length": 4, "crossfade": 10, "seed": 8}, files={"model": model, "chunks": vocabulary})
listen({"another take": job.output_bytes("result")})
print("remix:", progression(job.output_bytes("result")))

## Parameters

What to expect from each:

- `chunk_seconds`: the length of each chunk cut from `audio`. Cut on musical boundaries (a beat, a bar) and the result stays in time; anywhere else, notes get chopped.
- `length` (up to 500): how many chunks the output strings together.
- `variation` (above 0): how freely the result departs from the learned order. Low stays close to the source's own progression, high wanders.
- `quality`: how hard to train on a fresh piece: `instant` (no training, a quick shuffle), `fast`, `moderate` (default) or `complete` (slow and tight).
- `seed`: fixes a result. On a fresh piece it also fixes the learned reservoir; with a `model` it just picks a different take.
- `loop`: treat the piece as looping, so its end flows back into its start (learning only).
- `crossfade` (0 to 1000 ms): fades between chunks. A few milliseconds hides clicks at the joins; longer blurs them.
- `training_sequence`: advanced: chunk filenames in a specific order to learn instead of their natural order.

Inputs: `audio` to learn from, or `chunks` (a ZIP of WAVs, e.g. a previous `vocabulary`), plus `model` to skip training. Outputs: `result` (the WAV), `model`, `vocabulary` and `state`.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Tokens.** The input is cut into a vocabulary of tokens: here, `chunk_seconds`-long slices of the audio (the `vocabulary` output), and the piece becomes a sequence of them.

**Reservoir.** The sequence drives a quantum reservoir: a small, fixed quantum system whose state is nudged by each token in turn. Its measured features hold a fading memory of what came before, richer than the last token alone. The reservoir itself is never trained.

**Readout.** Only a simple readout is trained, mapping the reservoir's features to the next token. `quality` sets how hard it trains (`instant` skips training for a quick untrained shuffle). With `loop`, the end of the piece is treated as flowing back into its start.

**Generate.** Starting from the reservoir's state, the engine predicts the next token, samples one (`variation` sets how far it may stray from the most likely), feeds it back in and repeats `length` times.

The trained readout comes back as the `model` output: pass it into later runs to generate more takes without training again. The method follows *Level Generation with Quantum Reservoir Computing* (arXiv 2505.13287).

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Variation

Three 4-bar takes from the same trained model: from close to the loop's own progression to far from it.

In [ ]:
clips = {}
for variation in (0.2, 1.0, 3.0):
    job = client.jobs.run(engine=ENGINE, params={"length": 4, "variation": variation, "crossfade": 10, "seed": 1},
                          files={"model": model, "chunks": vocabulary}, quiet=True)
    clips[f"variation {variation}"] = job.output_bytes("result")
    print(f"variation={variation}:", progression(clips[f"variation {variation}"]))
listen(clips)

### Quality

Fresh training each time, 8 bars each: `instant` doesn't train, so its progression is close to a shuffle; `complete` follows the loop's own moves more closely.

In [ ]:
for quality in ("instant", "fast", "complete"):
    job = client.jobs.run(engine=ENGINE, params={"chunk_seconds": 2.5, "length": 8, "quality": quality, "loop": True, "seed": 7},
                          files={"audio": "loop.wav"}, quiet=True)
    print(f"{quality:>8}:", progression(job.output_bytes("result")))

### Chunk length

Half-bar chunks (1.25 s) let the reservoir split each bar: the halves recombine into new bars, so harmony can change mid-bar.

In [ ]:
job = client.jobs.run(engine=ENGINE, params={"chunk_seconds": 1.25, "length": 8, "quality": "fast", "loop": True, "crossfade": 10, "seed": 7},
                      files={"audio": "loop.wav"})
listen({"half-bar chunks": job.output_bytes("result")})

## Questions

**What makes this quantum?**

The memory: the reservoir is a quantum system, and its measured features, which carry the sequence's history, come from a quantum state much larger than its number of qubits suggests. Only the classical readout on top is trained.

**What kind of audio works best?**

Material that can be cut on musical boundaries: loops and grooves at a steady tempo, with `chunk_seconds` set to a beat or a bar. Each chunk is reused whole, so a boundary that falls mid-note will be heard as a cut.

**Can I reuse a model on different sounds?**

The model learns orderings of the vocabulary it was trained on. Pass the same `vocabulary` with it; a different ZIP of the same number of chunks would play new sounds in the learned order.

**Is this running on quantum hardware?**

No, the reservoir is simulated.

Sample sounds from Moth's QEcho sample pack (synthesised, no licence restrictions).